# Cross-Matching Astronomical Catalogs with LSDB and the Multimodal Universe

**Given a target RA/Dec, this notebook finds the nearest source in the HSC PDR3 and Legacy Survey catalogs — fetching only the sky pixels that overlap your search cone, not the full dataset.**

---

## What is LSDB?

[LSDB](https://lsdb.readthedocs.io) (Large Survey DataBase) is a Python library for efficiently querying and cross-matching large astronomical catalogs stored in **HATS format** (Hierarchical Adaptive Tiling Scheme). HATS partitions a catalog into HEALPix tiles on the sky, so operations like cone searches and cross-matches only touch the relevant tiles rather than scanning the full dataset.

The [Multimodal Universe](https://huggingface.co/UniverseTBD) project hosts dozens of survey catalogs in HATS format on Hugging Face, making them queryable directly from Python with no local download required.

---

## Catalogs Used

| Catalog | Dataset | Coverage | Bands |
|---|---|---|---|
| **HSC PDR3** | `UniverseTBD/mmu_hsc_pdr3_dud_22.5` | Deep+Ultra-Deep fields, i < 22.5 | g, r, i, z, y |
| **Legacy Survey North** | `UniverseTBD/mmu_ssl_legacysurvey_north` | Dec > +32.375° (MzLS + BASS) | g, r, z |
| **Legacy Survey South** | `hugging-science/mmu_legacysurvey_dr10_south_21` | Dec < +32.375° (DECaLS DR10) | g, r, i, z, W1–W4 |

There are many more catalogs available outside of these ones!

---

## Environment Setup

Create and activate the conda environment (only needed once):
```bash
conda create -n lsdb-env python=3.12 -y
conda activate lsdb-env
pip install lsdb huggingface-hub datasets pillow matplotlib jupyter
```

In future sessions, just activate:
```bash
conda activate lsdb-env
```

In [3]:
import lsdb
from astropy.coordinates import SkyCoord
import astropy.units as u
import numpy as np

In [4]:
TARGET_RA  =  335.0068 # degrees
TARGET_DEC =   -2.8368    # degrees

In [5]:
# Cone search radius used to filter both catalogs before cross-matching.
# Keep this small (30-60 arcsec) to avoid pulling large data chunks.
SEARCH_RADIUS_ARCSEC = 30.0

# Max separation allowed for a valid cross-match
XMATCH_RADIUS_ARCSEC = 1.0

In [6]:
HSC_CATALOG = "hf://datasets/UniverseTBD/mmu_hsc_pdr3_dud_22.5"
LS_NORTH_CATALOG  = "hf://datasets/UniverseTBD/mmu_ssl_legacysurvey_south"
LS_SOUTH_CATALOG = "hf://datasets/hugging-science/mmu_legacysurvey_dr10_south_21/"

HSC_COLUMNS = ["ra", "dec", "object_id",
                "g_cmodel_mag", "r_cmodel_mag", "i_cmodel_mag",
                "z_cmodel_mag", "y_cmodel_mag",
                "i_extendedness_value"]

LS_NORTH_COLUMNS  = ["ra", "dec", "object_id",
                "flux_g", "flux_r", "flux_z",
                "z_spec", "ebv"]

LS_SOUTH_COLUMNS  =["ra", "dec", "object_id",
               "FLUX_G", "FLUX_R", "FLUX_I", "FLUX_Z",
               "FLUX_W1", "FLUX_W2",
               "SHAPE_R", "SHAPE_E1", "SHAPE_E2",
               "EBV"]

In [7]:
def ls_flux_to_mag(flux):
    return 22.5-2.5*np.log10(flux)

In [8]:
coord = SkyCoord(ra=TARGET_RA * u.deg, dec=TARGET_DEC * u.deg)
print(f"Target: RA={TARGET_RA:.4f}, Dec={TARGET_DEC:+.4f}")
print(f"Cone search radius : {SEARCH_RADIUS_ARCSEC:.1f} arcsec")
print(f"Crossmatch radius  : {XMATCH_RADIUS_ARCSEC:.1f} arcsec")
print()

Target: RA=335.0068, Dec=-2.8368
Cone search radius : 30.0 arcsec
Crossmatch radius  : 1.0 arcsec



In [9]:
match_to_hsc = True
match_to_LS = True

if match_to_hsc==True:
    print('-'*60)
    print('-'*60)
    print('Matching to HSC')
    hsc = lsdb.open_catalog(HSC_CATALOG, columns=HSC_COLUMNS)
    hsc_flux_cols = ["g_cmodel_mag", "r_cmodel_mag", "i_cmodel_mag",
                     "z_cmodel_mag", "y_cmodel_mag"]
    print('Performing cone search.')
    hsc_cone = hsc.cone_search(TARGET_RA, TARGET_DEC, SEARCH_RADIUS_ARCSEC)

    
    result = hsc_cone.compute()
    
    sources = SkyCoord(ra=result["ra"].to_numpy()*u.deg, dec=result["dec"].to_numpy()*u.deg)
    separations = coord.separation(sources).arcsec
    
    try:
        closest = result.iloc[separations.argmin()]
        print(f'Closest source to target at RA={TARGET_RA}, Dec={TARGET_DEC}')
        print(f'is at RA={np.around(closest["ra"],4)}, Dec={np.around(closest["dec"],4)}, with separation {np.around(separations.min(),2)} arcsec:')
        print('Photometry:')
        for i,f in enumerate(closest[hsc_flux_cols]):
            print(f'{hsc_flux_cols[i]}: {np.around(f,2)} mag')
        
    except ValueError:
        print('No matches')
    print('-'*60)
    print('-'*60)

if match_to_LS==True:
    print('-'*60)
    print('-'*60)
    print('Matching to Legacy Survey')
    if TARGET_DEC > 32.375:
        print('Checking Northern catalog...')
        ls  = lsdb.open_catalog(LS_NORTH_CATALOG,  columns=LS_NORTH_COLUMNS)
        ls_flux_cols = ["flux_g", "flux_r", "flux_z"]
    else:
        print('Checking Southern catalog...')
        ls  = lsdb.open_catalog(LS_SOUTH_CATALOG,  columns=LS_SOUTH_COLUMNS)
        ls_flux_cols = ["FLUX_G", "FLUX_R", "FLUX_I", "FLUX_Z"]

    print('Performing cone search.')
    ls_cone  = ls.cone_search(TARGET_RA, TARGET_DEC, SEARCH_RADIUS_ARCSEC)
    result = ls_cone.compute()
    
    sources = SkyCoord(ra=result["ra"].to_numpy()*u.deg, dec=result["dec"].to_numpy()*u.deg)
    separations = coord.separation(sources).arcsec

    try:
        closest = result.iloc[separations.argmin()]
        print(f'Closest source to target at RA={TARGET_RA}, Dec={TARGET_DEC}')
        print(f'is at RA={np.around(closest["ra"],4)}, Dec={np.around(closest["dec"],4)}, with separation {np.around(separations.min(),2)} arcsec:')
        print('Photometry:')
        for i,f in enumerate(closest[ls_flux_cols]):
            print(f'{ls_flux_cols[i]}: {np.around(ls_flux_to_mag(f),2)} mag')
                
    except ValueError:
        print('No matches')

    print('-'*60)
    print('-'*60)

------------------------------------------------------------
------------------------------------------------------------
Matching to HSC
Performing cone search.


Computing Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

No matches
------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
------------------------------------------------------------
Matching to Legacy Survey
Checking Southern catalog...
Performing cone search.


Computing Catalog:   0%|          | 0/4 [00:00<?, ?it/s]

Closest source to target at RA=335.0068, Dec=-2.8368
is at RA=335.0068, Dec=-2.8368, with separation 0.19 arcsec:
Photometry:
FLUX_G: 22.58 mag
FLUX_R: 20.86 mag
FLUX_I: 20.26 mag
FLUX_Z: 19.93 mag
------------------------------------------------------------
------------------------------------------------------------
